In [1]:
# installing the depenendencies
!pip install unsloth
!pip install datasets

In [2]:
#get data set on google drive
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [ ]:
# Creating the architeture for models with unsloth compability

# carregando o dataset de treino
import pandas as pd
from datasets import Dataset

max_seq_length = 2048
model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
drive_path = '/content/drive/MyDrive/Data-Sets-Google-Coolabs/Classification-Models'

df = pd.read_json(drive_path + "/Train/training_data.json")

df_mapping = {"bom": 0, "médio": 1, "ruim": 2} # processo conhecido como Label Encoding (Mapeamento de classes)
df['label'] = df['classification'].map(df_mapping)

data_set = Dataset.from_pandas(df[['text_pt', 'label']])

Dataset({
    features: ['text_pt', 'label'],
    num_rows: 161
})


In [22]:
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer, SFTConfig

# carregar o modelo e realizar o treinamento

# como o modelo ja é quantizado em 4Bit, não precisa passar os parametros de quantização
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    full_finetuning=False,
    trust_remote_code=False
)

print("o modelo quantizado 4Bit foi carregado")

# adicionar os pesos de treinamento LoRA 
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    max_seq_length=max_seq_length,
    use_rslora=False,
    loftq_config=None
)

print("Pesos de treinamento LoRA foram adicionados")

==((====))==  Unsloth 2026.3.3: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-bnb-4bit as a legacy tokenizer.


o modelo quantizado 4Bit foi carregado
Pesos de treinamento LoRA foram adicionados


In [ ]:
# Treinamento do Modelo

def format_prompt_func(data):
    """
    Determina o formato de prompt para o modelo ser treinado.
    """
    report = data['text_pt']
    output = data['label']

    text = (
        f"Você é um Médico Profissional no Setor de Exames de Radiologia. "
        f"Analise este laudo e classifique-o:\n\n"
        f"Laudo: {report}\n\n"
        f"Classificação: {str(output)}"
    )

    return {"text": text}

data_set = data_set.map(format_prompt_func)

# 1. Garanta que o token de preenchimento existe
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 2. Configuração ajustada para maior estabilidade
trainer = SFTTrainer(
    model=model,
    train_dataset=data_set,
    dataset_text_field="text",
    tokenizer=tokenizer,
    args=SFTConfig(
        max_seq_length=max_seq_length,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,      # Aumentado para suavizar o gradiente
        warmup_steps=30,                    # Mantido proporcional (aprox 7% de 420)
        max_steps=420,
        learning_rate=5e-5,                 # Taxa de aprendizado conservadora para evitar saltos
        weight_decay=0.01,                  # Adiciona regularização para evitar overfitting
        logging_steps=5,                    # Log menos frequente para melhor leitura da tendência
        output_dir="outputs",
        optim="adamw_8bit",
        seed=3407,
        # Otimização adicional:
        lr_scheduler_type="cosine",         # Melhora a convergência final
    ),
)

trainer.train()


Map:   0%|          | 0/161 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=5):   0%|          | 0/161 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 161 | Num Epochs = 20 | Total steps = 420
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Step,Training Loss
1,2.634467
2,2.588095
3,2.726989
4,2.599456
5,2.661278
6,2.548983
7,2.287621
8,2.494324
9,2.601099
10,2.405174


In [ ]:
# Salvando o modelo

# Quando o treino termina, precisa exportar o modelo para que ele seja útil fora do ambiente de treino. -> salvar em formato GGUF é uma boa 

model.save_pretrained_gguf(
    "model_result",        # Nome da pasta onde será salvo o GGUF
    tokenizer,                 # Precisa do tokenizer para saber "falar"
    quantization_method="q4_k_m" # O método de compressão (balanceia qualidade/tamanho)
) 

# salvar em arquivos GGUF quando o treinamento do modelo for finalizado, ou seja, não é util para realizar um novo ajuste fino em modelos qauntizados em GGUF.

In [ ]:

test_dataset = load_dataset("json", data_files=drive_path + "/Test/testing_data.json") # load test_data 

In [34]:
FastLanguageModel.for_inference(model)

for d in test_dataset:
    inputs = tokenizer([d['text_pt']], return_tensors='pt').to('cuda')
    outputs = model.generate(**inputs, max_new_tokens=128)
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))